In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

shared_folder_path = '/content/drive/My Drive/relevance/filtered_relevance_race'

os.listdir(shared_folder_path)

In [ ]:
pip install supabase

In [ ]:
!pip -q install boto3 asyncpg python-dotenv nest_asyncio pandas

In [ ]:
pip install pyarrow s3fs


In [ ]:
# --- Load Supabase credentials from a local .env (gitignored); never hardcode secrets ---
import os
from pathlib import Path
_env = Path('.env')
if _env.exists():
    for _l in _env.read_text().splitlines():
        _l = _l.strip()
        if _l and not _l.startswith('#') and '=' in _l:
            _k, _v = _l.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
# Required env vars: SUPABASE_DB_URL, SUPABASE_S3_ENDPOINT, SUPABASE_S3_REGION,
#   SUPABASE_S3_ACCESS_KEY_ID, SUPABASE_S3_SECRET_ACCESS_KEY, SUPABASE_BUCKET

import os





In [ ]:
import os
import re
import io
import sys
import math
import asyncio
import concurrent.futures
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Tuple, Dict

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import asyncpg
import boto3
from boto3.s3.transfer import TransferConfig
from dotenv import load_dotenv

import pyarrow as pa
import pyarrow.parquet as pq

load_dotenv()

SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")

SUPABASE_S3_ENDPOINT = os.getenv("SUPABASE_S3_ENDPOINT")
SUPABASE_S3_REGION = os.getenv("SUPABASE_S3_REGION", "us-east-1")
SUPABASE_S3_ACCESS_KEY_ID = os.getenv("SUPABASE_S3_ACCESS_KEY_ID")
SUPABASE_S3_SECRET_ACCESS_KEY = os.getenv("SUPABASE_S3_SECRET_ACCESS_KEY")
SUPABASE_BUCKET = os.getenv("SUPABASE_BUCKET_NAME")

CHUNK_ROWS = int(os.getenv("CHUNK_ROWS", "80000"))
MAX_CONCURRENCY = int(os.getenv("MAX_CONCURRENCY", "4"))
MAX_OBJECT_MB = int(os.getenv("MAX_OBJECT_MB", "100"))
MAX_OBJECT_BYTES = MAX_OBJECT_MB * 1024 * 1024
ROW_GROUP_ROWS = int(os.getenv("ROW_GROUP_ROWS", "128000"))

REQUIRED = [
    ("SUPABASE_DB_URL", SUPABASE_DB_URL),
    ("SUPABASE_S3_ENDPOINT", SUPABASE_S3_ENDPOINT),
    ("SUPABASE_S3_ACCESS_KEY_ID", SUPABASE_S3_ACCESS_KEY_ID),
    ("SUPABASE_S3_SECRET_ACCESS_KEY", SUPABASE_S3_SECRET_ACCESS_KEY),
    ("SUPABASE_BUCKET_NAME", SUPABASE_BUCKET),
]
missing = [k for k, v in REQUIRED if not v]
if missing:
    raise SystemExit(f"Missing required env vars: {', '.join(missing)}")

boto_config = boto3.session.Config(
    retries={"max_attempts": 5, "mode": "adaptive"},
    signature_version="s3v4",
    max_pool_connections=max(10, MAX_CONCURRENCY * 4),
    connect_timeout=60,
    read_timeout=300,
)
s3_client = boto3.client(
    "s3",
    endpoint_url=SUPABASE_S3_ENDPOINT,
    aws_access_key_id=SUPABASE_S3_ACCESS_KEY_ID,
    aws_secret_access_key=SUPABASE_S3_SECRET_ACCESS_KEY,
    region_name=SUPABASE_S3_REGION,
    config=boto_config,
)
transfer_cfg = TransferConfig(
    multipart_threshold=5 * 1024 * 1024,
    multipart_chunksize=8 * 1024 * 1024,
    max_concurrency=min(MAX_CONCURRENCY, 10),
    use_threads=True,
)

DDL = """
CREATE TABLE IF NOT EXISTS metadata (
  id BIGSERIAL PRIMARY KEY,
  social_group TEXT NOT NULL,
  date DATE NOT NULL,                -- first day of month
  file_path TEXT NOT NULL,
  num_rows BIGINT NOT NULL,
  file_format TEXT,                  -- 'parquet' | 'csv' | 'csv.gz'
  size_bytes BIGINT,
  row_groups INT,
  ts_min TIMESTAMPTZ,
  ts_max TIMESTAMPTZ,
  year  INT GENERATED ALWAYS AS (EXTRACT(YEAR  FROM date)::int) STORED,
  month INT GENERATED ALWAYS AS (EXTRACT(MONTH FROM date)::int) STORED,
  CONSTRAINT file_format_chk CHECK (file_format IN ('parquet','csv','csv.gz'))
);
CREATE UNIQUE INDEX IF NOT EXISTS uniq_metadata_row
  ON metadata (social_group, date, file_path);
CREATE INDEX IF NOT EXISTS idx_metadata_group_date
  ON metadata (social_group, date);
CREATE INDEX IF NOT EXISTS idx_metadata_group_year_month
  ON metadata (social_group, year, month);
"""

async def create_db_pool(min_size: int = 1, max_size: int = 8) -> asyncpg.Pool:
    pool = await asyncpg.create_pool(
        SUPABASE_DB_URL,
        min_size=min_size,
        max_size=max_size,
        timeout=30,
    )
    async with pool.acquire() as conn:
        await conn.execute(DDL)
    return pool

async def upsert_metadata_with_retry(
    pool: asyncpg.Pool,
    group: str,
    num_rows: int,
    object_key: str,
    file_date: datetime.date,
    *,
    file_format: str = "parquet",
    size_bytes: Optional[int] = None,
    row_groups: Optional[int] = None,
    ts_min: Optional[datetime] = None,
    ts_max: Optional[datetime] = None,
    max_attempts: int = 5,
) -> None:

    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            async with pool.acquire() as conn:
                await conn.execute(
                    """
                    INSERT INTO metadata (social_group, num_rows, file_path, date,
                                           file_format, size_bytes, row_groups, ts_min, ts_max)
                    VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9)
                    ON CONFLICT (social_group, date, file_path)
                    DO UPDATE SET
                      num_rows   = EXCLUDED.num_rows,
                      file_format= EXCLUDED.file_format,
                      size_bytes = EXCLUDED.size_bytes,
                      row_groups = EXCLUDED.row_groups,
                      ts_min     = COALESCE(EXCLUDED.ts_min, metadata.ts_min),
                      ts_max     = COALESCE(EXCLUDED.ts_max, metadata.ts_max)
                    """,
                    group, num_rows, object_key, file_date,
                    file_format, size_bytes, row_groups, ts_min, ts_max
                )
            return
        except Exception as e:
            last_err = e
            await asyncio.sleep(min(2 ** attempt, 10))
    raise last_err

READ_CSV_KW = dict(
    usecols=[
        "id", "parent id", "text", "author", "time",
        "subreddit", "score", "matched patterns", "source_row"
    ],
    dtype={
        "score": "Int32",
        "source_row": "Int32",
        "id": "string",
        "parent id": "string",
        "text": "string",
        "author": "string",
        "time": "string",
        "subreddit": "string",
        "matched patterns": "string",
    },
    parse_dates=False,
    engine="c",
    low_memory=False,
)

FILENAME_MONTH_RE = re.compile(r"(\d{4}-\d{2})")

def deduce_file_month_date(filename: str) -> datetime.date:
    base = os.path.basename(filename)
    m = FILENAME_MONTH_RE.search(base)
    if not m:
        raise ValueError(f"Could not parse YYYY-MM from filename '{filename}'")
    return datetime.strptime(m.group(1) + "-01", "%Y-%m-%d").date()

def infer_social_group(path: Path) -> str:
    return path.parent.name.strip() or "ungrouped"

def parquet_object_key(group: str, filename: str, part_idx: int, sub_idx: Optional[int] = None) -> str:
    base = os.path.splitext(os.path.basename(filename))[0]
    if sub_idx is None:
        return f"corpus/{group}/{base}_part{part_idx}.parquet"
    else:
        return f"corpus/{group}/{base}_part{part_idx}_{sub_idx}.parquet"

def df_to_parquet_bytes(
    df: pd.DataFrame,
    *,
    row_group_rows: int = ROW_GROUP_ROWS,
    compression: str = "zstd",
) -> bytes:
    table = pa.Table.from_pandas(df, preserve_index=False)
    bio = io.BytesIO()
    pq.write_table(
        table,
        where=bio,
        compression=compression,
        use_dictionary=True,
        row_group_size=row_group_rows,
        coerce_timestamps="us",
        flavor="spark",
    )
    return bio.getvalue()

def parquet_bytes_meta(parquet_bytes: bytes) -> Tuple[int, Optional[int]]:
    size = len(parquet_bytes)
    try:
        pf = pq.ParquetFile(io.BytesIO(parquet_bytes))
        return size, pf.metadata.num_row_groups
    except Exception:
        return size, None

def split_df_to_parquet_parts_by_size(
    df: pd.DataFrame,
    max_bytes: int,
    *,
    row_group_rows: int = ROW_GROUP_ROWS,
) -> List[bytes]:
    parquet_bytes = df_to_parquet_bytes(df, row_group_rows=row_group_rows)
    if len(parquet_bytes) <= max_bytes or len(df) <= 1:
        return [parquet_bytes]

    mid = len(df) // 2
    left = split_df_to_parquet_parts_by_size(df.iloc[:mid], max_bytes, row_group_rows=row_group_rows)
    right = split_df_to_parquet_parts_by_size(df.iloc[mid:], max_bytes, row_group_rows=row_group_rows)
    return left + right

def extract_ts_min_max(df: pd.DataFrame) -> Tuple[Optional[datetime], Optional[datetime]]:
    if "time" not in df.columns:
        return None, None
    try:
        dt = pd.to_datetime(df["time"], errors="coerce", utc=True)
        if dt.notna().any():
            return dt.min().to_pydatetime(), dt.max().to_pydatetime()
    except Exception:
        pass
    return None, None

async def upload_parquet_and_record(
    pool: asyncpg.Pool,
    group: str,
    src_filename: str,
    part_idx: int,
    sub_idx: Optional[int],
    parquet_bytes: bytes,
    rows: int,
    file_date: datetime.date,
    ts_bounds: Tuple[Optional[datetime], Optional[datetime]],
    row_groups: Optional[int],
    loop: asyncio.AbstractEventLoop,
    executor: concurrent.futures.Executor,
) -> Tuple[str, int]:
    object_key = parquet_object_key(group, src_filename, part_idx, sub_idx)
    size_mb = len(parquet_bytes) / (1024 * 1024)
    print(f"[UPLOAD] {object_key} | {rows} rows | {size_mb:.2f} MB | row_groups={row_groups}")

    def _upload():
        s3_client.upload_fileobj(
            Fileobj=io.BytesIO(parquet_bytes),
            Bucket=SUPABASE_BUCKET,
            Key=object_key,
            ExtraArgs={
                "ContentType": "application/octet-stream",
                "Metadata": {
                    "rows": str(rows),
                    "date": str(file_date),
                    "group": group,
                    "format": "parquet",
                },
            },
            Config=transfer_cfg,
        )

    await loop.run_in_executor(executor, _upload)

    size_bytes = len(parquet_bytes)
    ts_min, ts_max = ts_bounds

    await upsert_metadata_with_retry(
        pool,
        group,
        rows,
        object_key,
        file_date,
        file_format="parquet",
        size_bytes=size_bytes,
        row_groups=row_groups,
        ts_min=ts_min,
        ts_max=ts_max,
    )
    return object_key, rows

async def process_single_file(
    pool: asyncpg.Pool,
    loop: asyncio.AbstractEventLoop,
    executor: concurrent.futures.Executor,
    file_path: Path,
    group_override: Optional[str],
    CHUNK_ROWS: int,
    MAX_OBJECT_BYTES: int,
    sem: asyncio.Semaphore,
) -> int:
    if not file_path.exists() or not file_path.is_file():
        print(f"[WARN] Skipping (missing/not a file): {file_path}")
        return 0
    try:
        file_date = deduce_file_month_date(str(file_path))
    except ValueError as e:
        print(f"[WARN] {e} (file: {file_path})")
        return 0

    group = group_override or infer_social_group(file_path)
    part_idx = 0
    file_total = 0
    tasks: List[asyncio.Task] = []

    print(f"[INFO] Processing: {file_path}  group={group}  date={file_date}  "
          f"chunk_rows={CHUNK_ROWS}  max_object={MAX_OBJECT_MB}MB parquet")

    try:
        for chunk in pd.read_csv(str(file_path), chunksize=CHUNK_ROWS, **READ_CSV_KW):
            part_idx += 1

            ts_bounds = extract_ts_min_max(chunk)

            parquet_parts = split_df_to_parquet_parts_by_size(
                chunk,
                MAX_OBJECT_BYTES,
                row_group_rows=ROW_GROUP_ROWS,
            )
            print(f"[SPLIT] {file_path.name} part {part_idx}: {len(parquet_parts)} parquet sub-parts")

            total_bytes = sum(len(b) for b in parquet_parts)
            total_rows = len(chunk)

            def est_rows_for_blob(blob_len: int) -> int:
                if total_bytes > 0:
                    return max(1, round(total_rows * (blob_len / total_bytes)))
                return max(1, total_rows // max(1, len(parquet_parts)))

            sub_idx = 0
            for blob in parquet_parts:
                sub_idx += 1
                size_bytes, rg = parquet_bytes_meta(blob)
                rows_est = est_rows_for_blob(size_bytes)

                await sem.acquire()
                async def _do_upload(
                    parquet_bytes=blob,
                    rows=rows_est,
                    pidx=part_idx,
                    sidx=sub_idx,
                    tsb=ts_bounds,
                    row_groups=rg,
                ):
                    nonlocal file_total
                    try:
                        key, r = await upload_parquet_and_record(
                            pool, group, str(file_path), pidx, sidx,
                            parquet_bytes, rows, file_date, tsb, row_groups,
                            loop, executor
                        )
                        file_total += r
                        print(f"[OK] {key}")
                    except Exception as e:
                        print(f"[ERR] Upload/record failed ({file_path.name} part {pidx}_{sidx}): {e}")
                    finally:
                        sem.release()

                tasks.append(asyncio.create_task(_do_upload()))
        if tasks:
            await asyncio.gather(*tasks)
    except Exception as e:
        print(f"[ERR] Failed to read {file_path}: {e}")
        return file_total

    print(f"[DONE] {file_path.name} → ~{file_total:,} rows uploaded (estimated for split parts)")
    return file_total

async def process_files_by_path(
    FILE_PATHS: List[str],
    SINGLE_GROUP: Optional[str] = None,
    CHUNK_ROWS: int = CHUNK_ROWS,
    MAX_CONCURRENCY: int = MAX_CONCURRENCY,
    MAX_OBJECT_BYTES: int = MAX_OBJECT_BYTES,
):
    loop = asyncio.get_running_loop()
    pool = await create_db_pool(min_size=1, max_size=max(4, MAX_CONCURRENCY * 2))

    executor = concurrent.futures.ThreadPoolExecutor(
        max_workers=max(2, min(os.cpu_count() or 4, 16))
    )
    pool_cap = boto_config.max_pool_connections or 10
    eff_conc = min(MAX_CONCURRENCY, max(1, pool_cap // 2))
    sem = asyncio.Semaphore(eff_conc)

    print(f"[CFG] MAX_CONCURRENCY requested={MAX_CONCURRENCY}  effective={eff_conc}  "
          f"max_pool_connections={pool_cap}")

    total_uploaded = 0
    try:
        for p in FILE_PATHS:
            path_obj = Path(p).expanduser().resolve()
            rows = await process_single_file(
                pool=pool,
                loop=loop,
                executor=executor,
                file_path=path_obj,
                group_override=SINGLE_GROUP,
                CHUNK_ROWS=CHUNK_ROWS,
                MAX_OBJECT_BYTES=MAX_OBJECT_BYTES,
                sem=sem,
            )
            total_uploaded += rows
    finally:
        await pool.close()
        executor.shutdown(wait=True)

    print(f"\nAll uploads complete. Total rows uploaded (approx across splits): {total_uploaded:,}")

async def repair_missing_metadata_by_prefix(
    prefix: str,
    group: str,
    default_date: Optional[datetime.date] = None,
):
    pool = await create_db_pool(min_size=1, max_size=4)

    keys: List[str] = []
    token = None
    while True:
        kwargs = {"Bucket": SUPABASE_BUCKET, "Prefix": prefix}
        if token:
            kwargs["ContinuationToken"] = token
        resp = s3_client.list_objects_v2(**kwargs)
        for c in resp.get("Contents", []):
            if c["Key"].endswith(".parquet"):
                keys.append(c["Key"])
        if resp.get("IsTruncated"):
            token = resp.get("NextContinuationToken")
        else:
            break

    print(f"[REPAIR] Found {len(keys)} objects under {prefix}")

    async with pool.acquire() as conn:
        existing = await conn.fetch(
            "SELECT file_path FROM metadata WHERE social_group=$1 AND file_path=ANY($2::text[])",
            group, keys
        )
    already = {r["file_path"] for r in existing}
    missing = [k for k in keys if k not in already]
    print(f"[REPAIR] Missing rows to insert: {len(missing)}")

    for k in missing:
        head = s3_client.head_object(Bucket=SUPABASE_BUCKET, Key=k)
        meta = head.get("Metadata", {}) or {}
        rows = int(meta.get("rows", "-1"))
        date_str = meta.get("date")
        fdate = datetime.strptime(date_str, "%Y-%m-%d").date() if date_str else (default_date or datetime.utcnow().date())
        size_bytes = head.get("ContentLength", None)

        try:
            await upsert_metadata_with_retry(
                pool, group, rows, k, fdate,
                file_format="parquet",
                size_bytes=size_bytes,
                row_groups=None, ts_min=None, ts_max=None
            )
            print(f"[REPAIR OK] {k} ({rows})")
        except Exception as e:
            print(f"[REPAIR ERR] {k}: {e}")

    await pool.close()


files = [
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2009-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2010-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2012-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2013-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2015-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2017-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2019-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2021-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-05.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-08.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-10.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2023-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-04.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2007-01.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2014-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-12.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2022-06.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2008-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2020-03.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-07.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2016-02.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2011-09.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-11.csv",
    "/content/drive/My Drive/relevance/filtered_relevance_race/RC_2018-05.csv"
]
await process_files_by_path(
    FILE_PATHS=files,
    SINGLE_GROUP="race",
    CHUNK_ROWS=80000,
    MAX_CONCURRENCY=4,
    MAX_OBJECT_BYTES=100*1024*1024,
)

# await repair_missing_metadata_by_prefix(
#     prefix="corpus/race/RC_2023-03",
#     group="race",
# )